# InsureWise AI - Production Data Preprocessing Pipeline Construction
**Notebook:** `03_preprocessing.ipynb`  
**Project:** InsureWise AI - Explainable ML for Personalized Insurance Premium Prediction  
**Objective:** Construct a leak-free, modular, production-ready preprocessing pipeline using scikit-learn's `Pipeline` and `ColumnTransformer` for the selected primary dataset (`medical_insurance_cost_dataset_enhanced.csv`). Export the fitted pipeline structure (`preprocessing_pipeline.joblib`) for production deployment via FastAPI.

---

## Table of Contents
1. [Environment Setup & Data Loading](#1.-Environment-Setup-&-Data-Loading)
2. [Feature & Target Separation (X & y)](#2.-Feature-&-Target-Separation-(X-&-y))
3. [Train-Test Splitting (Data Leakage Prevention)](#3.-Train-Test-Splitting-(Data-Leakage-Prevention))
4. [Custom Feature Transformer: BloodPressureExtractor](#4.-Custom-Feature-Transformer:-BloodPressureExtractor)
5. [Automated Feature Type Identification](#5.-Automated-Feature-Type-Identification)
6. [Outlier Investigation & Strategy Justification](#6.-Outlier-Investigation-&-Strategy-Justification)
7. [Modular Sub-Pipelines & ColumnTransformer Construction](#7.-Modular-Sub-Pipelines-&-ColumnTransformer-Construction)
8. [End-to-End Pipeline Fitting, Transformation & Test Verification](#8.-End-to-End-Pipeline-Fitting,-Transformation-&-Test-Verification)
9. [Artifact Export & FastAPI Serving Simulation](#9.-Artifact-Export-&-FastAPI-Serving-Simulation)
10. [Preprocessing Decisions & Compliance Summary](#10.-Preprocessing-Decisions-&-Compliance-Summary)

> **Execution Rules & Constraints:**
> - Primary Dataset: `ml-service/data/raw/medical_insurance_cost_dataset_enhanced.csv` (7,500 rows, 22 columns).
> - Raw data files are kept strictly unchanged.
> - Data split into Train/Test BEFORE fitting any preprocessing transformation to avoid data leakage.
> - Preprocessing implemented via scikit-learn `Pipeline` and `ColumnTransformer`. No manual separate train/test preprocessing.
> - Pipeline exported to `preprocessing_pipeline.joblib` for FastAPI deployment.

In [1]:
import os
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore')

# Set aesthetic styling for visualizations
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.4f' % x)

print("Environment initialized successfully.")
print(f"Pandas version: {pd.__version__}")
print(f"Numpy version: {np.__version__}")
print(f"Joblib version: {joblib.__version__}")

Environment initialized successfully.
Pandas version: 3.0.6
Numpy version: 2.5.3
Joblib version: 1.6.0


## 1. Environment Setup & Data Loading

We load the primary dataset selected during Data Understanding and EDA (`medical_insurance_cost_dataset_enhanced.csv`), located at `../data/raw/medical_insurance_cost_dataset_enhanced.csv`. The dataset contains 7,500 rows and 22 columns spanning clinical, lifestyle, demographic, and financial variables.

In [2]:
# Define data path with fallback support
data_path = '../data/raw/medical_insurance_cost_dataset_enhanced.csv'
if not os.path.exists(data_path):
    data_path = 'd:/Ml_Project/Ml_Service/Data/Raw/medical_insurance_cost_dataset_enhanced.csv'

df_raw = pd.read_csv(data_path)
print(f"Primary dataset loaded successfully from: {data_path}")
print(f"Dimensions: {df_raw.shape[0]} rows, {df_raw.shape[1]} columns")
print(f"Missing values across entire dataset: {df_raw.isnull().sum().sum()}")
print(f"Duplicate rows: {df_raw.duplicated().sum()}")
df_raw.head(3)

Primary dataset loaded successfully from: d:/Ml_Project/Ml_Service/Data/Raw/medical_insurance_cost_dataset_enhanced.csv
Dimensions: 7500 rows, 22 columns
Missing values across entire dataset: 0
Duplicate rows: 0


## 2. Feature & Target Separation (`X` & `y`)

We separate the raw dataset into the feature matrix `X` and target vector `y`:
- **Target Variable (`y`):** `annual_medical_cost_usd` (continuous regression target).
- **Identifier Column (`customer_id`):** Primary key identifier (`"MIC100001"`). Non-predictive string ID dropped from feature matrix `X` to prevent arbitrary identifier memorization.
- **Predictor Features (`X`):** 20 remaining features including raw composite string column `blood_pressure` (`"SYS/DIA"`).

In [3]:
target_col = 'annual_medical_cost_usd'
id_col = 'customer_id'

# Separate features X and target y
X = df_raw.drop(columns=[target_col, id_col])
y = df_raw[target_col]

print("=== FEATURE & TARGET SEPARATION ===")
print(f"Target column 'y': {target_col} | Shape: {y.shape}")
print(f"Dropped identifier: {id_col}")
print(f"Feature matrix 'X' shape: {X.shape} ({X.shape[1]} predictor features)")
print("\nPredictor Feature Columns:")
for i, col in enumerate(X.columns, 1):
    print(f"  {i:2d}. {col:30s} (dtype: {X[col].dtype})")

=== FEATURE & TARGET SEPARATION ===
Target column 'y': annual_medical_cost_usd | Shape: (7500,)
Dropped identifier: customer_id
Feature matrix 'X' shape: (7500, 20) (20 predictor features)

Predictor Feature Columns:
   1. age                            (dtype: int64)
   2. gender                         (dtype: str)
   3. bmi                            (dtype: float64)
   4. children                       (dtype: int64)
   5. smoker                         (dtype: str)
   6. region                         (dtype: str)
   7. occupation                     (dtype: str)
   8. annual_income_usd              (dtype: int64)
   9. exercise_level                 (dtype: str)
  10. chronic_diseases               (dtype: int64)
  11. doctor_visits_per_year         (dtype: int64)
  12. hospitalizations_last_year     (dtype: int64)
  13. alcohol_consumption_per_week   (dtype: int64)
  14. insurance_plan                 (dtype: str)
  15. blood_pressure                 (dtype: str)
  16. diabetes 

## 3. Train-Test Splitting (Data Leakage Prevention)

> **CRITICAL DATA LEAKAGE REQUIREMENT:**  
> Splitting data into training and test sets **MUST** occur **BEFORE** fitting any preprocessing transformations (such as scaling parameters $\mu, \sigma$, imputers, or categorical encoders). Fitting transformers on the entire dataset before splitting causes information from the test set to leak into the training process, resulting in overly optimistic evaluation metrics.

We split the dataset into an **80% Training Set** (6,000 samples) and a **20% Test Set** (1,500 samples) using a fixed seed (`random_state=42`). The test set is kept strictly unseen until final model evaluation.

In [4]:
# Split data into training and test sets BEFORE fitting any preprocessor
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("=== TRAIN-TEST SPLIT SUMMARY ===")
print(f"Training Features (X_train): {X_train.shape[0]} rows, {X_train.shape[1]} columns")
print(f"Testing Features (X_test):   {X_test.shape[0]} rows, {X_test.shape[1]} columns")
print(f"Training Target (y_train):   {y_train.shape[0]} rows (Mean: ${y_train.mean():,.2f}, Std: ${y_train.std():,.2f})")
print(f"Testing Target (y_test):     {y_test.shape[0]} rows (Mean: ${y_test.mean():,.2f}, Std: ${y_test.std():,.2f})")

=== TRAIN-TEST SPLIT SUMMARY ===
Training Features (X_train): 6000 rows, 20 columns
Testing Features (X_test):   1500 rows, 20 columns
Training Target (y_train):   6000 rows (Mean: $17,908.45, Std: $9,655.12)
Testing Target (y_test):     1500 rows (Mean: $17,869.83, Std: $9,712.44)


## 4. Custom Feature Transformer: `BloodPressureExtractor`

The feature `blood_pressure` is formatted as a composite string (`"Systolic/Diastolic"`, e.g. `"149/87"`). Downstream machine learning models require numerical inputs.

To ensure **100% compatibility with scikit-learn Pipelines** and seamless **FastAPI REST API serving**, we write a custom scikit-learn transformer class `BloodPressureExtractor` inheriting from `BaseEstimator` and `TransformerMixin`:
1. Parses string `"SYS/DIA"` into numeric integer/float columns `systolic_bp` and `diastolic_bp`.
2. Drops the raw string column `blood_pressure`.
3. Handles both pandas DataFrames and raw dictionary payloads from FastAPI API requests without raising errors.
4. Implements `get_feature_names_out()` for full compatibility with `ColumnTransformer` feature tracking.

In [5]:
class BloodPressureExtractor(BaseEstimator, TransformerMixin):
    """
    Scikit-learn custom transformer to parse composite 'blood_pressure' string ("SYS/DIA")
    into numeric 'systolic_bp' and 'diastolic_bp' features.
    
    Compatible with pandas DataFrames, dicts, and FastAPI JSON request payloads.
    """
    def __init__(self, bp_col='blood_pressure'):
        self.bp_col = bp_col

    def fit(self, X, y=None):
        # Stateless transformer
        return self

    def transform(self, X):
        # Ensure DataFrame copy
        if isinstance(X, pd.DataFrame):
            X_out = X.copy()
        else:
            X_out = pd.DataFrame(X).copy()

        if self.bp_col in X_out.columns:
            # Split SYS/DIA
            split_bp = X_out[self.bp_col].astype(str).str.split('/', expand=True)
            X_out['systolic_bp'] = pd.to_numeric(split_bp[0], errors='coerce').astype('float64')
            X_out['diastolic_bp'] = pd.to_numeric(split_bp[1], errors='coerce').astype('float64')
            X_out = X_out.drop(columns=[self.bp_col])
        return X_out

    def get_feature_names_out(self, input_features=None):
        if input_features is None:
            return None
        feature_names = [f for f in input_features if f != self.bp_col]
        feature_names.extend(['systolic_bp', 'diastolic_bp'])
        return np.array(feature_names)

# Test BloodPressureExtractor on X_train sample
bp_test_transformer = BloodPressureExtractor()
X_train_bp_sample = bp_test_transformer.transform(X_train.head(3))

print("=== CUSTOM TRANSFORMER (BloodPressureExtractor) VERIFICATION ===")
print("Original blood_pressure sample:", list(X_train['blood_pressure'].head(3)))
print("Extracted systolic_bp sample:  ", list(X_train_bp_sample['systolic_bp']))
print("Extracted diastolic_bp sample: ", list(X_train_bp_sample['diastolic_bp']))
print("Columns after extraction:", list(X_train_bp_sample.columns))

=== CUSTOM TRANSFORMER (BloodPressureExtractor) VERIFICATION ===
Original blood_pressure sample: ['149/87', '131/74', '126/76']
Extracted systolic_bp sample:   [149.0, 131.0, 126.0]
Extracted diastolic_bp sample:  [87.0, 74.0, 76.0]
Columns after extraction: ['age', 'gender', 'bmi', 'children', 'smoker', 'region', 'occupation', 'annual_income_usd', 'exercise_level', 'chronic_diseases', 'doctor_visits_per_year', 'hospitalizations_last_year', 'alcohol_consumption_per_week', 'insurance_plan', 'diabetes', 'cholesterol', 'sleep_hours', 'stress_level', 'marital_status', 'systolic_bp', 'diastolic_bp']


## 5. Automated Feature Type Identification

After parsing `blood_pressure` into numerical `systolic_bp` and `diastolic_bp`, the dataset contains 21 features. We programmatically identify and classify features into three categories:

1. **Numerical Features (13 features):** Continuous and discrete numeric predictors (`age`, `bmi`, `children`, `annual_income_usd`, `chronic_diseases`, `doctor_visits_per_year`, `hospitalizations_last_year`, `alcohol_consumption_per_week`, `cholesterol`, `sleep_hours`, `stress_level`, `systolic_bp`, `diastolic_bp`).
2. **Ordinal Categorical Features (2 features):** Ordered categorical variables (`exercise_level`: `['Low', 'Moderate', 'High']`; `insurance_plan`: `['Basic', 'Standard', 'Premium', 'Gold']`).
3. **Nominal Categorical Features (6 features):** Unordered categorical variables (`gender`, `smoker`, `region`, `occupation`, `diabetes`, `marital_status`).

In [6]:
# Apply BP extractor to inspect training features
X_train_parsed = bp_test_transformer.transform(X_train)

# Programmatically detect numerical features
num_features = X_train_parsed.select_dtypes(include=['int64', 'float64']).columns.tolist()

# Define explicit ordinal categories and columns
ordinal_features = ['exercise_level', 'insurance_plan']
ordinal_category_orders = [
    ['Low', 'Moderate', 'High'],
    ['Basic', 'Standard', 'Premium', 'Gold']
]

# Identify remaining nominal categorical features automatically
categorical_all = X_train_parsed.select_dtypes(include=['object', 'str', 'category']).columns.tolist()
nominal_features = [col for col in categorical_all if col not in ordinal_features]

print("=== AUTOMATED FEATURE CLASSIFICATION ===")
print(f"Total Numerical Features ({len(num_features)}):")
print(" ", num_features)
print(f"\nTotal Ordinal Features ({len(ordinal_features)}):")
print(" ", ordinal_features)
print(f"\nTotal Nominal Features ({len(nominal_features)}):")
print(" ", nominal_features)

=== AUTOMATED FEATURE CLASSIFICATION ===
Total Numerical Features (13):
  ['age', 'bmi', 'children', 'annual_income_usd', 'chronic_diseases', 'doctor_visits_per_year', 'hospitalizations_last_year', 'alcohol_consumption_per_week', 'cholesterol', 'sleep_hours', 'stress_level', 'systolic_bp', 'diastolic_bp']

Total Ordinal Features (2):
  ['exercise_level', 'insurance_plan']

Total Nominal Features (6):
  ['gender', 'smoker', 'region', 'occupation', 'diabetes', 'marital_status']


## 6. Outlier Investigation & Strategy Justification

We quantitatively analyze numerical outliers within `X_train` using both **IQR (Interquartile Range)** and **Z-Score ($|Z| > 3.0$)** rules to make explicit, domain-justified decisions.

In [7]:
outlier_records = []

for col in num_features:
    s = X_train_parsed[col]
    q1 = s.quantile(0.25)
    q3 = s.quantile(0.75)
    iqr = q3 - q1
    lower_iqr = q1 - 1.5 * iqr
    upper_iqr = q3 + 1.5 * iqr
    iqr_count = ((s < lower_iqr) | (s > upper_iqr)).sum()
    
    mean_val = s.mean()
    std_val = s.std()
    z_count = (np.abs((s - mean_val) / std_val) > 3.0).sum() if std_val > 0 else 0
    
    outlier_records.append({
        'Feature': col,
        'Min': s.min(),
        'Max': s.max(),
        'IQR_Lower': lower_iqr,
        'IQR_Upper': upper_iqr,
        'IQR_Outliers': iqr_count,
        'IQR_%': (iqr_count / len(X_train)) * 100,
        'Z_Outliers(>3.0)': z_count,
        'Z_%': (z_count / len(X_train)) * 100
    })

outlier_df = pd.DataFrame(outlier_records)
print("=== NUMERICAL OUTLIER ANALYSIS ON TRAINING DATA (X_train) ===")
print(outlier_df.to_string(index=False))

=== NUMERICAL OUTLIER ANALYSIS ON TRAINING DATA (X_train) ===
                   Feature       Min         Max   IQR_Lower   IQR_Upper  IQR_Outliers  IQR_%  Z_Outliers(>3.0)    Z_%
                       age   18.0000     65.0000     -6.5000     93.5000             0 0.0000                 0 0.0000
                       bmi   15.0000     50.0000      6.6000     57.0000             0 0.0000                 0 0.0000
                  children    0.0000      5.0000     -3.0000      5.0000             0 0.0000                 0 0.0000
         annual_income_usd 15000.0000 150000.0000 -27847.3750 193309.6250             0 0.0000                 0 0.0000
          chronic_diseases    0.0000      4.0000     -1.5000      2.5000           207 3.4500                 0 0.0000
    doctor_visits_per_year    0.0000     12.0000     -4.5000     11.5000            90 1.5000                 0 0.0000
hospitalizations_last_year    0.0000      3.0000      0.0000      0.0000            15 0.2500           

### Outlier Handling Decisions & Rationale:

1. **Target Variable (`annual_medical_cost_usd`):**  
   - The EDA revealed 83 target outliers (1.11%) with costs exceeding $40,517.  
   - **Decision:** **Do NOT trim or truncate target outliers.** Medical insurance claims naturally exhibit right-skewed heavy tails due to catastrophic care events. Truncating target outliers would systematically underprice insurance premiums for high-risk policyholders, destroying underwriting solvency.
2. **Feature Outliers (`chronic_diseases`, `hospitalizations_last_year`, `doctor_visits_per_year`):**  
   - Outliers represent authentic clinical states (e.g. 3 hospitalizations or 4 chronic diseases).  
   - **Decision:** Preserve all observations and apply **`StandardScaler`** (or `RobustScaler` compatible imputers) within the numerical feature pipeline. Standard scaling centers features around mean zero with unit variance, stabilizing gradient descent without discarding valid patient risk profiles.

---

## 7. Modular Sub-Pipelines & `ColumnTransformer` Construction

We construct three specialized preprocessing sub-pipelines:

1. **Numerical Pipeline:**
   - `SimpleImputer(strategy='median')`: Handles missing numerical values using training median.
   - `StandardScaler()`: Scales continuous and discrete numerical features to zero mean and unit variance ($\mu=0, \sigma=1$).
2. **Ordinal Pipeline:**
   - `SimpleImputer(strategy='most_frequent')`: Imputes missing categorical values using training mode.
   - `OrdinalEncoder(categories=[['Low', 'Moderate', 'High'], ['Basic', 'Standard', 'Premium', 'Gold']], handle_unknown='use_encoded_value', unknown_value=-1)`: Preserves ordered integer relationships.
3. **Nominal Pipeline:**
   - `SimpleImputer(strategy='most_frequent')`: Imputes missing categorical values.
   - `OneHotEncoder(handle_unknown='ignore', sparse_output=False)`: Creates binary indicator columns ($0/1$) for nominal features, ignoring unknown categories during inference.

We assemble these using `ColumnTransformer` and wrap everything with `BloodPressureExtractor` inside a master `Pipeline`.

In [8]:
# 1. Numerical Sub-Pipeline
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# 2. Ordinal Categorical Sub-Pipeline
ord_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OrdinalEncoder(
        categories=ordinal_category_orders,
        handle_unknown='use_encoded_value',
        unknown_value=-1
    ))
])

# 3. Nominal Categorical Sub-Pipeline
nom_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(
        handle_unknown='ignore',
        sparse_output=False
    ))
])

# Assemble ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_pipeline, num_features),
        ('ord', ord_pipeline, ordinal_features),
        ('nom', nom_pipeline, nominal_features)
    ],
    remainder='drop'
)

# Master Pipeline wrapping custom BloodPressureExtractor & ColumnTransformer
full_pipeline = Pipeline([
    ('bp_extractor', BloodPressureExtractor()),
    ('preprocessor', preprocessor)
])

print("=== SCALING APPLICABILITY & PIPELINE ARCHITECTURE ===")
print("1. Scaling applied ONLY to numerical features (StandardScaler).")
print("2. Scaling NOT applied to One-Hot binary encoded variables (preserving 0/1 indicator interpretation).")
print("3. Master Pipeline constructed with 2 stages: [bp_extractor -> preprocessor].")

=== SCALING APPLICABILITY & PIPELINE ARCHITECTURE ===
1. Scaling applied ONLY to numerical features (StandardScaler).
2. Scaling NOT applied to One-Hot binary encoded variables (preserving 0/1 indicator interpretation).
3. Master Pipeline constructed with 2 stages: [bp_extractor -> preprocessor].


## 8. End-to-End Pipeline Fitting, Transformation & Test Verification

We fit the master `full_pipeline` **STRICTLY ON `X_train`** using `fit_transform(X_train)`.  
We then transform the completely unseen test set `X_test` using `transform(X_test)`.

> **Verification:** `fit()` is NEVER called on `X_test`. All scaling means, standard deviations, and encoder vocabularies are learned solely from `X_train` to avoid data leakage.

In [9]:
# Fit master pipeline strictly on X_train and transform X_train
X_train_processed = full_pipeline.fit_transform(X_train)

# Transform test set X_test using fitted pipeline (NO RE-FITTING)
X_test_processed = full_pipeline.transform(X_test)

# Retrieve transformed feature names
feature_names_out = full_pipeline.named_steps['preprocessor'].get_feature_names_out()

print("=== PIPELINE EXECUTION & FEATURE MATRIX METRICS ===")
print(f"X_train original shape:   {X_train.shape}")
print(f"X_train processed shape:  {X_train_processed.shape} (6,000 samples, {X_train_processed.shape[1]} features)")
print(f"X_test original shape:    {X_test.shape}")
print(f"X_test processed shape:   {X_test_processed.shape} (1,500 samples, {X_test_processed.shape[1]} features)")
print(f"Total encoded features:   {len(feature_names_out)}")

print("\nSample Feature Names (First 15 features):")
for i, fname in enumerate(feature_names_out[:15], 1):
    print(f"  {i:2d}. {fname}")

# Verify training numerical features have mean ~ 0 and std ~ 1
num_means = X_train_processed[:, :len(num_features)].mean(axis=0)
num_stds = X_train_processed[:, :len(num_features)].std(axis=0)
print(f"\nTraining Numerical Scaled Means Range: [{num_means.min():.6f}, {num_means.max():.6f}] (Expected ~ 0)")
print(f"Training Numerical Scaled Stds Range:  [{num_stds.min():.6f}, {num_stds.max():.6f}] (Expected ~ 1)")

=== PIPELINE EXECUTION & FEATURE MATRIX METRICS ===
X_train original shape:   (6000, 20)
X_train processed shape:  (6000, 41) (6,000 samples, 41 features)
X_test original shape:    (1500, 20)
X_test processed shape:   (1500, 41) (1,500 samples, 41 features)
Total encoded features:   41

Sample Feature Names (First 15 features):
   1. num__age
   2. num__bmi
   3. num__children
   4. num__annual_income_usd
   5. num__chronic_diseases
   6. num__doctor_visits_per_year
   7. num__hospitalizations_last_year
   8. num__alcohol_consumption_per_week
   9. num__cholesterol
  10. num__sleep_hours
  11. num__stress_level
  12. num__systolic_bp
  13. num__diastolic_bp
  14. ord__exercise_level
  15. ord__insurance_plan

Training Numerical Scaled Means Range: [-0.000000, 0.000000] (Expected ~ 0)
Training Numerical Scaled Stds Range:  [1.000000, 1.000000] (Expected ~ 1)


## 9. Artifact Export & FastAPI Serving Simulation

To ensure that downstream model training (`04_model_training.ipynb`) and the **FastAPI REST API** use the exact same preprocessing transformations:

1. **Pipeline Export:** Save `full_pipeline` to `../models/preprocessing_pipeline.joblib`.
2. **Processed Datasets Export:** Save `X_train_processed`, `X_test_processed`, `y_train`, `y_test`, and `feature_names.json` to `../data/processed/`.
3. **FastAPI Payload Simulation:** Load `preprocessing_pipeline.joblib` and pass a raw JSON payload containing string `"blood_pressure": "135/85"` to verify that `.transform()` operates seamlessly.

In [10]:
# Define export directories
models_dir = '../models'
processed_data_dir = '../data/processed'

if not os.path.exists(models_dir):
    models_dir = 'd:/Ml_Project/Ml_Service/models'
if not os.path.exists(processed_data_dir):
    processed_data_dir = 'd:/Ml_Project/Ml_Service/Data/Processed'

os.makedirs(models_dir, exist_ok=True)
os.makedirs(processed_data_dir, exist_ok=True)

# 1. Save fitted pipeline artifact
pipeline_export_path = os.path.join(models_dir, 'preprocessing_pipeline.joblib')
joblib.dump(full_pipeline, pipeline_export_path)
print(f"Fitted preprocessing pipeline successfully saved to: {pipeline_export_path}")

# 2. Save preprocessed DataFrames & target vectors
df_X_train_proc = pd.DataFrame(X_train_processed, columns=feature_names_out)
df_X_test_proc = pd.DataFrame(X_test_processed, columns=feature_names_out)

df_X_train_proc.to_csv(os.path.join(processed_data_dir, 'X_train.csv'), index=False)
df_X_test_proc.to_csv(os.path.join(processed_data_dir, 'X_test.csv'), index=False)
y_train.to_csv(os.path.join(processed_data_dir, 'y_train.csv'), index=False)
y_test.to_csv(os.path.join(processed_data_dir, 'y_test.csv'), index=False)

with open(os.path.join(processed_data_dir, 'feature_names.json'), 'w') as f:
    json.dump(list(feature_names_out), f, indent=2)

print(f"Preprocessed train/test datasets successfully saved to: {processed_data_dir}")

# 3. FASTAPI REST API INFERENCE SIMULATION
print("\n=== FASTAPI REST API INFERENCE SIMULATION ===")

# Simulate incoming raw REST API request payload (JSON dict)
fastapi_sample_payload = pd.DataFrame([{
    'age': 52,
    'gender': 'Female',
    'bmi': 31.2,
    'children': 1,
    'smoker': 'Yes',
    'region': 'Southwest',
    'occupation': 'Engineer',
    'annual_income_usd': 92000,
    'exercise_level': 'High',
    'chronic_diseases': 2,
    'doctor_visits_per_year': 5,
    'hospitalizations_last_year': 1,
    'alcohol_consumption_per_week': 3,
    'insurance_plan': 'Gold',
    'blood_pressure': '138/88',
    'diabetes': 'Yes',
    'cholesterol': 225,
    'sleep_hours': 6.5,
    'stress_level': 5.0,
    'marital_status': 'Single'
}])

# Load pipeline from joblib
loaded_fastapi_pipeline = joblib.load(pipeline_export_path)

# Transform raw API payload
processed_payload = loaded_fastapi_pipeline.transform(fastapi_sample_payload)

print(f"Loaded Pipeline Type: {type(loaded_fastapi_pipeline)}")
print(f"Raw Input Payload shape: {fastapi_sample_payload.shape}")
print(f"Transformed Output shape: {processed_payload.shape}")
print(f"Successfully converted raw 'blood_pressure' ('138/88') to numeric features in 41-dim feature vector.")
print("FastAPI serving integration readiness: VERIFIED SUCCESSFUL!")

Fitted preprocessing pipeline successfully saved to: d:/Ml_Project/Ml_Service/models/preprocessing_pipeline.joblib
Preprocessed train/test datasets successfully saved to: d:/Ml_Project/Ml_Service/Data/Processed

=== FASTAPI REST API INFERENCE SIMULATION ===
Loaded Pipeline Type: <class 'sklearn.pipeline.Pipeline'>
Raw Input Payload shape: (1, 20)
Transformed Output shape: (1, 41)
Successfully converted raw 'blood_pressure' ('138/88') to numeric features in 41-dim feature vector.
FastAPI serving integration readiness: VERIFIED SUCCESSFUL!


## 10. Preprocessing Decisions & Compliance Summary

The following verification matrix details how each of the 12 project requirements was strictly satisfied:

| Requirement | Implementation Strategy | Verification Status |
| :--- | :--- | :--- |
| **1. Separate X and y** | `y = annual_medical_cost_usd`, `X = df.drop(columns=['annual_medical_cost_usd', 'customer_id'])` | Verified |
| **2. Feature Identification** | Programmatic classification: 13 Numerical, 2 Ordinal, 6 Nominal features | Verified |
| **3. Missing Values Strategy** | `SimpleImputer(strategy='median')` for numeric; `SimpleImputer(strategy='most_frequent')` for categoricals | Verified |
| **4. Categorical Encoding** | `OrdinalEncoder` for ordered ranks; `OneHotEncoder(handle_unknown='ignore')` for nominals | Verified |
| **5. Feature Scaling** | `StandardScaler` applied ONLY to numerical features; non-scaled OHE binary matrices | Verified |
| **6. Outlier Handling** | Domain justification: High medical cost target tails ($> $40k) preserved; numeric scaling used | Verified |
| **7. Data Leakage Prevention** | `train_test_split(test_size=0.2)` performed BEFORE fitting pipeline | Verified |
| **8. Pre-fit Splitting** | `X_train` (6,000 samples) and `X_test` (1,500 samples) created prior to `pipeline.fit()` | Verified |
| **9. Pipeline Architecture** | Scikit-learn `Pipeline` + `ColumnTransformer` + custom `BloodPressureExtractor` | Verified |
| **10. Single Unified Transformation** | No separate manual code; `fit_transform(X_train)` and `transform(X_test)` used | Verified |
| **11. Unseen Test Set** | Test set kept completely untouched during `fit()`; evaluated strictly via `transform()` | Verified |
| **12. Decision Explanation** | Detailed Markdown sections explaining mathematical and domain rationale for every decision | Verified |
| **FastAPI Export** | `preprocessing_pipeline.joblib` saved and validated with simulated REST API payload | Verified |

---
> **Verification Check:** Raw datasets were not modified. No machine learning models were trained or tuned in this notebook per project instructions.